# Compare two Texas Hold'em hands

Run the setup cell first, then run the input cell. Enter cards using notation such as `As`, `Kd`, and either `Th` or `10h`.

The result is from the first hand's perspective:

- `0` = loss
- `1` = tie
- `2` = win
- `incompatible` = invalid input or a card appears more than once

In [ ]:
import sys
from pathlib import Path

project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "cards.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from cards import hand_names, make_hand
from showdown import PRIVATE_HANDS, compare_hands, comparison_matrix, comparison_vector

def compare_from_text(first_text, second_text, river_text):
    try:
        first = make_hand(first_text.split())
        second = make_hand(second_text.split())
        river = make_hand(river_text.split())
        return compare_hands(first, second, river)
    except (TypeError, ValueError):
        return "incompatible"

PRIVATE_HAND_LABELS = [" ".join(hand_names(hand)) for hand in PRIVATE_HANDS]

def vector_from_text(private_text, river_text):
    private = make_hand(private_text.split())
    river = make_hand(river_text.split())
    return comparison_vector(private, river)

def matrix_from_text(river_text):
    river = make_hand(river_text.split())
    return comparison_matrix(river)

print("Ready to compare hands.")

## Enter the cards

Edit the three quoted values in the cell below, then run the cell.

## Vector and matrix comparisons

Both axes use `PRIVATE_HAND_LABELS`, a stable list of all 1,326 private hands. `I` means the two hands overlap or a hand overlaps the board.

In [ ]:
private = "10c Ad"
river = "2c 7d 9h Js 3c"

vector = vector_from_text(private, river)
matrix = matrix_from_text(river)

print("Vector entries:", len(vector))
print("Matrix shape:", len(matrix), "x", len(matrix[0]))

# Example lookup: compare the private hand above with 9c Kd.
opponent_index = PRIVATE_HAND_LABELS.index("9c Kd")
print("Against 9c Kd:", vector[opponent_index])

## Visualise the vector and matrix

Run the comparison cell above first. If Matplotlib is missing, install the development requirements with `python -m pip install -r requirements-dev.txt`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Convert I, 0, 1, and 2 to consecutive numeric colour values.
colour_value = {"I": 0, 0: 1, 1: 2, 2: 3}
colours = ListedColormap(["#6b7280", "#dc2626", "#f59e0b", "#16a34a"])
legend_items = [
    Patch(color="#6b7280", label="I: incompatible"),
    Patch(color="#dc2626", label="0: loss"),
    Patch(color="#f59e0b", label="1: tie"),
    Patch(color="#16a34a", label="2: win"),
]

vector_colours = [[colour_value[value] for value in vector]]
fig, ax = plt.subplots(figsize=(15, 2.2))
ax.imshow(vector_colours, aspect="auto", interpolation="nearest", cmap=colours, vmin=0, vmax=3)
ax.set_title(f"{private} against every possible private hand")
ax.set_xlabel("Opponent hand index in PRIVATE_HAND_LABELS")
ax.set_yticks([])
ax.legend(handles=legend_items, loc="upper center", bbox_to_anchor=(0.5, -0.45), ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
matrix_colours = [[colour_value[value] for value in row] for row in matrix]
fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(matrix_colours, interpolation="nearest", cmap=colours, vmin=0, vmax=3)
ax.set_title(f"All private-hand comparisons on board {river}")
ax.set_xlabel("Column hand index")
ax.set_ylabel("Row hand index (result perspective)")

# Label a small number of evenly spaced hands so the axes stay readable.
tick_indexes = list(range(0, len(PRIVATE_HAND_LABELS), 200))
ax.set_xticks(tick_indexes, [PRIVATE_HAND_LABELS[i] for i in tick_indexes], rotation=45, ha="right")
ax.set_yticks(tick_indexes, [PRIVATE_HAND_LABELS[i] for i in tick_indexes])
ax.legend(handles=legend_items, loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=4)
plt.tight_layout()
plt.show()

In [ ]:
# Change these three values, keeping each value inside quotes.
first_text = "Jc Ad"
second_text = "9c Kd"
river_text = "2c 7d 9h Js 3c"

result = compare_from_text(first_text, second_text, river_text)
print("Result:", result)

## Examples without prompts

You can also edit the strings below and rerun the cell.

In [ ]:
first = "10c Ad"
second = "9c Kd"
river = "2c 7d 9h Js 3c"

print(compare_from_text(first, second, river))  # 0: first hand loses

In [ ]:
# This is incompatible because As appears in both private hands.
print(compare_from_text("As Kd", "As Qd", "2c 7d 9h Js 3c"))